<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/alex/RA2_Trabajo_pr%C3%A1ctico_N%C2%B0_5_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP5 — Ejercicio 2: Clasificación de especies de Iris

**Inteligencia Computacional · IC415 · Año 2026**

---

## Contexto del problema

Un jardín botánico nacional dispone de 150 registros morfológicos (largo y ancho de sépalo y pétalo) de tres especies del género *Iris*: *setosa*, *versicolor* y *virginica*. Se busca un sistema automático de identificación de especie a partir de esas cuatro mediciones.

La pregunta directiva no es solo "construir una RNA que clasifique", sino:

> ¿Es la red neuronal la herramienta adecuada para este caso, o un clasificador más simple alcanza los mismos resultados con menor complejidad?

Esa pregunta determina el recorrido del notebook: no apuntamos a un único número de accuracy, sino a sostener una decisión de ingeniería con evidencia experimental.

## El dato más crítico

**150 muestras, 50 por clase.** Ese número condiciona absolutamente todo lo que sigue. Con 80/20 estratificado quedan 120 muestras de entrenamiento y 30 de test (10 por clase). Un solo error de clasificación en test mueve la accuracy un 3,3%. Esto significa que la varianza entre ejecuciones será alta y que **comparar dos modelos por décimas de accuracy carece de significancia estadística**. Lo trataremos explícitamente en la sección de estabilidad entre semillas.

## Estructura del notebook

1. Setup, semillas y reproducibilidad.
2. Análisis Exploratorio (EDA).
3. Preprocesamiento y pipeline.
4. Modelado con RNA: arquitectura base, experimentos de activación y tamaño, modelo final con callbacks.
5. Evaluación de la RNA, estabilidad entre semillas.
6. Comparación con modelos clásicos (LazyClassifier + profundización).
7. Visualización de fronteras de decisión.
8. Curva de aprendizaje: ¿estamos en el techo del problema?
9. Conclusiones y respuestas a las consignas.

> **Nota de ejecución.** El notebook está pensado para Google Colab (12 GB de RAM, 100 GB de almacenamiento). Iris es un dataset trivial en términos de cómputo, así que no hay restricciones de memoria que considerar. Cada celda se ejecuta de principio a fin sin dependencias externas no declaradas. Las semillas se fijan explícitamente para que los resultados sean idénticos en cada corrida.


## 1. Setup, imports y semillas

Fijamos las semillas globales **antes de cualquier import que las use**. Esto cubre:

- `random` (Python puro).
- `numpy.random` (sklearn lo usa internamente).
- `tensorflow` y `keras` (RNA).
- `os.environ['PYTHONHASHSEED']` (orden de iteración de diccionarios y hashing).
- `tf.config.experimental.enable_op_determinism()` cuando esté disponible (operaciones deterministas en TF).

Sin esto, dos ejecuciones consecutivas dan métricas distintas y la comparación entre arquitecturas pierde toda validez.


In [ ]:
# Fijación de semillas ANTES de cualquier import que las use.
# La variable de entorno debe definirse antes de importar TensorFlow.
import os
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)

# TensorFlow / Keras
import tensorflow as tf
tf.random.set_seed(SEED)
try:
    # Disponible desde TF 2.8; refuerza determinismo en CUDA.
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# Función helper para re-fijar todas las semillas en un punto del notebook.
# Se usa en el experimento de estabilidad entre semillas.
def fijar_semillas(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Semilla global: {SEED}")


In [ ]:
# Imports del resto de la pila.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn import datasets
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, learning_curve
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Modelos clásicos para la comparación
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
)

# Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# Configuración estética
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', '{:.3f}'.format)


## 2. Análisis Exploratorio (EDA)

### ¿Por qué hacer EDA aunque el dataset sea conocido?

Iris es un dataset "de juguete", limpio, balanceado, sin nulos. Aun así, el EDA no es opcional: es la base argumentativa de las decisiones posteriores. Concretamente, queremos confirmar empíricamente:

1. **Que el dataset está limpio** (sin nulos, tipos correctos).
2. **Que las clases están balanceadas** (50/50/50).
3. **Que las features tienen escalas diferentes** (justifica el uso de `StandardScaler`).
4. **Que existe separabilidad geométrica entre clases** (justifica que arquitecturas pequeñas alcancen).
5. **Qué pares de clases son más difíciles de separar** (anticipa los errores en la matriz de confusión).


In [ ]:
# Carga del dataset Iris desde sklearn.
iris = datasets.load_iris()

# Construimos un DataFrame para EDA.
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))

print("Forma del dataset:", df.shape)
df.head()


In [ ]:
# Tipos, nulos, memoria.
df.info()


In [ ]:
# Estadísticos descriptivos por feature.
df.drop(columns=['target']).describe()


**Lectura inicial.**

- 150 filas, 5 columnas (4 features + target). Sin nulos.
- `sepal length` y `petal length` se miden en la misma unidad (cm) pero con rangos muy distintos: el sépalo va de 4.3 a 7.9, el pétalo de 1.0 a 6.9. La desviación de `petal length` es más del doble que la de `sepal width`.
- **Implicancia para la RNA:** una neurona tipo `relu` con pesos inicializados He Normal espera entradas con varianza similar entre features. Si no escalamos, la feature de mayor varianza domina los gradientes en las primeras épocas y la red converge mucho más lento. Esto justifica el `StandardScaler` posterior.


In [ ]:
# Distribución de clases.
conteo = df['species'].value_counts().sort_index()
print(conteo)
print("\nProporción:")
print((conteo / len(df) * 100).round(1).astype(str) + ' %')


Balanceo perfecto (33.3% por clase). Esto significa que:

- **Accuracy es una métrica honesta** (no hay clase mayoritaria que infle la métrica con predicciones triviales).
- No necesitamos `class_weight`, ni oversampling, ni umbrales especiales.
- En la división estratificada, mantener la proporción es trivial: cada clase aporta exactamente 10 muestras al test set 80/20.


In [ ]:
# Pairplot: la herramienta de diagnóstico más informativa para Iris.
# Mostrar las 4 features cruzadas, coloreadas por especie.
pp = sns.pairplot(
    df.drop(columns=['target']),
    hue='species',
    diag_kind='kde',
    plot_kws={'alpha': 0.7, 's': 35, 'edgecolor': 'k', 'linewidth': 0.3},
    height=2.0,
    palette='viridis'
)
pp.fig.suptitle('Pairplot Iris — separabilidad por especie', y=1.02, fontsize=13)
plt.show()


**Lectura crítica del pairplot — ESTO es lo que define el resto del notebook.**

1. **Iris setosa es linealmente separable** del resto en CUALQUIER par que incluya `petal length` o `petal width`. Una sola línea recta basta. Un perceptrón simple sin capas ocultas resuelve esa separación. Esto explica por qué cualquier modelo (incluso uno muy simple) va a tener cero errores con setosa.

2. **Versicolor y virginica se solapan parcialmente** en la zona intermedia, especialmente en `sepal length` y `sepal width`. La separación más limpia entre estas dos clases vive en el plano (`petal length`, `petal width`). Es ahí donde se concentrarán los errores de clasificación.

3. **`petal length` y `petal width` son las features más informativas.** Las distribuciones por clase están claramente separadas. `sepal length` y `sepal width`, en cambio, presentan distribuciones bastante solapadas.

**Hipótesis derivadas:**

- *H1.* Cualquier modelo razonable alcanzará accuracy ≥ 90%, porque setosa es trivial (33% gratis) y versicolor/virginica son separables casi linealmente.
- *H2.* Los errores de clasificación se concentrarán en versicolor↔virginica, no en setosa. La matriz de confusión debe reflejarlo.
- *H3.* Una RNA pequeña va a alcanzar el techo del problema. Una RNA grande va a sobreajustar y, en el mejor caso, igualar a una pequeña.


In [ ]:
# Boxplots por feature, agrupados por especie, para ver visualmente
# qué features separan mejor las clases.
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
features = iris.feature_names
for ax, f in zip(axes, features):
    sns.boxplot(data=df, x='species', y=f, ax=ax, palette='viridis', hue='species', legend=False)
    ax.set_title(f)
    ax.set_xlabel('')
plt.suptitle('Distribución de cada feature por especie', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


Los boxplots confirman lo que vimos en el pairplot: `petal length` y `petal width` discriminan netamente las tres clases (las cajas casi no se superponen). `sepal width` apenas separa setosa del resto y deja indistinguibles a versicolor y virginica. **Si tuviéramos que entrenar un clasificador con una sola feature, sería `petal length` o `petal width`.**


In [ ]:
# Matriz de correlación entre las 4 features numéricas.
corr = df[features].corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
            square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de correlación de features')
plt.show()


**Observaciones de la correlación:**

- `petal length` y `petal width` están **muy correlacionadas** (~0.96). Son casi la misma información medida de dos formas. Para una RNA esto no es un problema, pero indica que el dataset, en términos de información independiente, tiene **menos de 4 dimensiones efectivas**.
- `sepal length` correlaciona también con las dos de pétalo (~0.87 y ~0.82), porque las flores grandes tienden a serlo en todos sus aspectos.
- `sepal width` es la única feature relativamente independiente. También es la menos discriminante.

**Implicancia:** la dimensionalidad efectiva del problema es muy baja. Una arquitectura con 4 entradas y 16 neuronas ocultas ya tiene una capacidad de representación holgada para este problema.


## 3. Preprocesamiento y pipeline

### 3.1 Split estratificado 80/20

El TP permite 80/20 o 70/30. Elegimos **80/20 estratificado** porque:

- Con 150 muestras, dejar más datos para train ayuda a la red (que tiene más parámetros que el dataset es chico).
- `stratify=y` garantiza 10 muestras por clase en test. Sin estratificar, una mala suerte podría dejarnos con 7-13 en alguna clase y cualquier métrica por-clase sería ruidosa.
- Los modelos clásicos los evaluaremos también con **5-fold CV estratificada** para tener una estimación menos dependiente del split particular.


In [ ]:
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print("\nDistribución por clase en train:")
print(pd.Series(y_train).value_counts().sort_index().to_dict())
print("\nDistribución por clase en test:")
print(pd.Series(y_test).value_counts().sort_index().to_dict())


### 3.2 Escalado dentro de un Pipeline (prevención de Data Leakage)

El escalado es **crítico** para una RNA: las funciones de activación tipo sigmoid/tanh saturan con entradas grandes, y ReLU con He Normal asume entradas con media 0 y varianza ~1. Pero hay una trampa:

> Si calculamos media y desvío con TODO el dataset y después dividimos en train/test, **el test set "filtra" información hacia el train**. La media usada para escalar train ya conoce la distribución de test. Esto se llama **Data Leakage** y produce métricas artificialmente optimistas.

La forma correcta es:

1. Dividir train/test primero.
2. `fit` del scaler **solo** con train.
3. `transform` aplicado a train y a test con los parámetros aprendidos en train.

Encapsulamos esto en un `sklearn.Pipeline` para que el flujo quede inmutable y reproducible. Para los modelos clásicos esto se vuelve obligatorio cuando hagamos cross-validation: si escaláramos antes del CV, todos los folds estarían contaminados.


In [ ]:
# Escalamos las features para la RNA.
# Para los clásicos, el escalado se incluirá dentro de un Pipeline por modelo.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verificación: media ~0 y desvío ~1 en train, valores cercanos en test.
print("Train escalado — media por feature:", X_train_scaled.mean(axis=0).round(3))
print("Train escalado — std por feature: ", X_train_scaled.std(axis=0).round(3))
print("Test  escalado — media por feature:", X_test_scaled.mean(axis=0).round(3))
print("Test  escalado — std por feature: ", X_test_scaled.std(axis=0).round(3))


### 3.3 Codificación de la salida — dos formas válidas

El TP pregunta explícitamente sobre las opciones de codificación. Ambas dan **resultados matemáticamente idénticos** (la pérdida calcula lo mismo); cambia solo la forma del tensor de etiquetas.

| Codificación | `y` | Loss |
|---|---|---|
| Label encoding | `[0, 1, 2, 0, ...]` shape `(N,)` | `sparse_categorical_crossentropy` |
| One-Hot encoding | `[[1,0,0],[0,1,0],[0,0,1],[1,0,0]...]` shape `(N, 3)` | `categorical_crossentropy` |

**Criterio de ingeniería:** usar `sparse_categorical_crossentropy` cuando el target ya es entero (ahorra memoria y un paso de transformación), y reservar one-hot para cuando se necesita explícitamente trabajar con probabilidades por clase (por ejemplo, para *label smoothing* o *soft labels*). Aquí no hay ninguna razón para preferir una sobre otra. Vamos a usar **label encoding** por simplicidad, pero mostramos que el resultado coincide.

La capa de salida es **siempre** `Dense(3, activation='softmax')`. Las 3 neuronas representan las 3 clases; softmax normaliza las salidas a probabilidades que suman 1.


In [ ]:
# Demostración rápida: las dos codificaciones llevan a la misma loss.
# Reconstruimos cada y para mostrar la equivalencia.
ohe = OneHotEncoder(sparse_output=False)
y_train_oh = ohe.fit_transform(y_train.reshape(-1, 1))
y_test_oh = ohe.transform(y_test.reshape(-1, 1))

print("Label encoding y_train[:5]:", y_train[:5])
print("\nOne-Hot y_train[:5]:")
print(y_train_oh[:5].astype(int))
print(f"\nShapes: label {y_train.shape} vs one-hot {y_train_oh.shape}")


## 4. Modelado con Red Neuronal Artificial

### 4.1 Arquitectura base — justificación del tamaño

Antes de probar nada, fijamos un orden de magnitud razonable. Con 4 features de entrada y 3 de salida, una red `[16, 8]` (dos capas ocultas) tiene aproximadamente:

- Capa 1: 4·16 + 16 = **80 parámetros**
- Capa 2: 16·8 + 8 = **136 parámetros**
- Salida: 8·3 + 3 = **27 parámetros**

**Total: ~243 parámetros**, frente a 120 muestras de entrenamiento. La razón parámetros/muestras es ~2:1, ya alta. Cualquier red más grande aumenta el riesgo de overfitting de forma desproporcionada.

**Activación oculta:** ReLU. Es el estándar de la industria desde 2012 por dos razones:
- Su derivada es 1 en la zona positiva, lo que evita el *vanishing gradient* que sufren sigmoid y tanh en redes profundas.
- Computacionalmente es trivial (un `max(0, x)`).

**Inicialización:** He Normal. Está diseñada específicamente para ReLU y mantiene la varianza de las activaciones estable a lo largo de las capas.

**Activación de salida:** softmax (3 clases, probabilidades que suman 1).

**Optimizador:** Adam con `learning_rate=1e-3`. Adam adapta la tasa de aprendizaje por parámetro y es robusto sin tuning fino. Para un problema chico como este, casi cualquier optimizador funcionaría, pero Adam es el default sensato.


In [ ]:
def construir_modelo(arquitectura=(16, 8), activacion='relu',
                      input_dim=4, n_clases=3, l2_reg=0.0, dropout_rate=0.0):
    """Construye un MLP secuencial con la arquitectura indicada.

    Parameters
    ----------
    arquitectura : tuple of int
        Cantidad de neuronas por capa oculta. Ej: (16, 8) -> 2 capas.
    activacion : str
        Función de activación de las capas ocultas.
    l2_reg : float
        Coeficiente L2 (0 = sin regularización).
    dropout_rate : float
        Tasa de dropout post-cada capa oculta (0 = sin dropout).
    """
    capas = [Input(shape=(input_dim,))]
    for n in arquitectura:
        capas.append(Dense(
            n, activation=activacion,
            kernel_initializer='he_normal' if activacion == 'relu' else 'glorot_uniform',
            kernel_regularizer=l2(l2_reg) if l2_reg > 0 else None
        ))
        if dropout_rate > 0:
            capas.append(Dropout(dropout_rate))
    capas.append(Dense(n_clases, activation='softmax'))
    modelo = Sequential(capas)
    modelo.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return modelo


# Probamos la arquitectura base y mostramos el summary.
fijar_semillas(SEED)
modelo_base = construir_modelo(arquitectura=(16, 8), activacion='relu')
modelo_base.summary()


El `summary` confirma el conteo manual: la red tiene **243 parámetros entrenables**. Esta es nuestra arquitectura candidata; los experimentos siguientes la cuestionan.

### 4.2 Callbacks: piloto automático del entrenamiento

Dos callbacks centrales para evitar tanto entrenar de menos como entrenar de más:

- **`EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)`** detiene el entrenamiento cuando `val_loss` no mejora durante 20 épocas seguidas, y devuelve los pesos del momento de mejor `val_loss` (no del final, que ya estaría sobreajustado).
- **`ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10)`** baja a la mitad la tasa de aprendizaje cuando se estanca por 10 épocas. Permite a la red asentarse en un mínimo local más fino.

La relación `patience(LR)=10 < patience(Stop)=20` es intencional: queremos que ReduceLR intervenga *antes* de que EarlyStopping aborte, para darle a la red la chance de mejorar con un LR más chico.


In [ ]:
def construir_callbacks(patience_stop=20, patience_lr=10):
    return [
        EarlyStopping(monitor='val_loss', patience=patience_stop,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=patience_lr, min_lr=1e-6, verbose=0)
    ]


### 4.3 Entrenamiento del modelo base y lectura de las curvas

Entrenamos `[16, 8]` ReLU durante un máximo de 200 épocas con un 20% del train como validación. Esperamos que EarlyStopping corte mucho antes.


In [ ]:
fijar_semillas(SEED)
modelo_base = construir_modelo(arquitectura=(16, 8), activacion='relu')

t0 = time.time()
hist_base = modelo_base.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=16,
    callbacks=construir_callbacks(),
    verbose=0
)
t_train = time.time() - t0

ep_finales = len(hist_base.history['loss'])
val_acc_final = hist_base.history['val_accuracy'][-1]
print(f"Entrenamiento finalizado en {ep_finales} epocas ({t_train:.2f} s).")
print(f"Validation accuracy en la mejor epoca (restaurada): {val_acc_final:.4f}")


In [ ]:
def graficar_historia(historias, titulo='Curvas de entrenamiento'):
    """Grafica loss y accuracy (train vs val) de una o varias historias.

    historias : dict {label: hist} o un único History.
    """
    if not isinstance(historias, dict):
        historias = {'modelo': historias}

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    cmap = plt.get_cmap('tab10')

    for i, (label, h) in enumerate(historias.items()):
        c = cmap(i)
        ep = range(1, len(h.history['loss']) + 1)
        axes[0].plot(ep, h.history['loss'], color=c, linestyle='-',
                     label=f'{label} - train')
        axes[0].plot(ep, h.history['val_loss'], color=c, linestyle='--',
                     label=f'{label} - val')
        axes[1].plot(ep, h.history['accuracy'], color=c, linestyle='-',
                     label=f'{label} - train')
        axes[1].plot(ep, h.history['val_accuracy'], color=c, linestyle='--',
                     label=f'{label} - val')

    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Accuracy')
    for ax in axes:
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3)
    plt.suptitle(titulo, fontsize=13)
    plt.tight_layout()
    plt.show()


graficar_historia(hist_base, 'Modelo base [16, 8] - ReLU')


**Lectura de las curvas (modelo base):**

Para sostener que el modelo "no está ni sobreajustado ni sub-ajustado", buscamos tres signos en las curvas:

1. **Loss de train y validation deben ambas bajar** (descartar underfitting / mala configuración).
2. **La brecha entre train_loss y val_loss debe permanecer pequeña** (descartar overfitting).
3. **La accuracy de validation debe estabilizarse en una meseta alta**, sin caer al final.

Si la curva de validación empieza a subir mientras la de train sigue bajando, hay overfitting. Si ambas se estancan en valores altos de loss, hay underfitting (la arquitectura es insuficiente o el LR está mal).


### 4.4 Experimento A — Funciones de activación

El TP pregunta: *"¿Todas las funciones de activación disponibles permiten llegar a un resultado aceptable? ¿Cuáles presentan problemas y por qué?"*

Comparamos cinco activaciones manteniendo todo lo demás fijo (arquitectura `[16, 8]`, optimizador, semilla):

- **ReLU:** estándar moderno. Derivada 1 en zona positiva, 0 en negativa.
- **Tanh:** salida en [-1, 1], centrada en 0, derivada saturable.
- **Sigmoid:** salida en [0, 1], no centrada, derivada saturable, vanishing gradient.
- **ELU:** alternativa a ReLU; suaviza la zona negativa, evita "dying ReLU".
- **Leaky ReLU:** otra alternativa, con pendiente pequeña en la zona negativa.

> **Nota sobre la "trampa" del experimento.** Para que la comparación sea limpia, NO ajustamos `kernel_initializer` por activación: usamos `glorot_uniform` para todas excepto ReLU (que usa He Normal, su pareja teórica). Si forzáramos He Normal con sigmoid, estaríamos induciendo un fallo que no es de la activación sino del par activación-inicialización.


In [ ]:
# Para Leaky ReLU manejamos compatibilidad entre Keras 2 (alpha) y Keras 3
# (negative_slope), y también el caso de que la activación no exista como string.
import tensorflow.keras.activations as kact
from tensorflow.keras.layers import LeakyReLU

def _leaky_relu_layer():
    """Devuelve una capa LeakyReLU compatible con la versión de Keras instalada."""
    try:
        return LeakyReLU(negative_slope=0.01)  # Keras 3
    except TypeError:
        return LeakyReLU(alpha=0.01)           # Keras 2

def modelo_con_activacion(act_name):
    """Devuelve un modelo [16, 8] con la activación pedida."""
    if act_name == 'leaky_relu':
        modelo = Sequential([
            Input(shape=(4,)),
            Dense(16, kernel_initializer='glorot_uniform'),
            _leaky_relu_layer(),
            Dense(8, kernel_initializer='glorot_uniform'),
            _leaky_relu_layer(),
            Dense(3, activation='softmax')
        ])
        modelo.compile(optimizer=Adam(1e-3),
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])
        return modelo
    return construir_modelo(arquitectura=(16, 8), activacion=act_name)


activaciones = ['relu', 'tanh', 'sigmoid', 'elu', 'leaky_relu']
historias_act = {}
tiempos_act = {}

for act in activaciones:
    fijar_semillas(SEED)  # misma semilla para todas: comparación limpia
    modelo = modelo_con_activacion(act)
    t0 = time.time()
    h = modelo.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        epochs=200,
        batch_size=16,
        callbacks=construir_callbacks(),
        verbose=0
    )
    tiempos_act[act] = time.time() - t0
    historias_act[act] = h
    eps = len(h.history['loss'])
    print(f"{act:12s}  epocas={eps:3d}  val_acc={h.history['val_accuracy'][-1]:.4f}  "
          f"val_loss={h.history['val_loss'][-1]:.4f}  tiempo={tiempos_act[act]:.1f}s")


In [ ]:
graficar_historia(historias_act, 'Experimento A - Comparación de activaciones')


**Lectura del Experimento A.**

Lo que esperamos ver y por qué:

- **ReLU, ELU y Leaky ReLU convergen rápido y a buen accuracy.** Su derivada vale 1 (o ~1) en la zona donde la mayoría de los pesos viven, y eso permite que el gradiente fluya sin atenuación. Las tres son de la "familia rectificada" y se comportan parecido en problemas chicos.

- **Tanh converge bien pero suele necesitar más épocas.** Su derivada máxima vale 1 (en x=0) y cae rápido a 0. Para una red de 2 capas no es catastrófico; en redes profundas sí lo sería.

- **Sigmoid es la peor del lote.** Su derivada máxima vale 0.25 (en x=0), y cae a casi 0 fuera de un rango muy estrecho. Esto significa que durante el backpropagation el gradiente se atenúa por capa, fenómeno conocido como *vanishing gradient*. En una red [16, 8] con sólo 2 capas el efecto es leve, pero ya se nota: convergencia más lenta y/o accuracy final menor. En redes profundas (10+ capas) sigmoid es directamente inviable. Por eso se reservó para la **capa de salida en clasificación binaria** (donde su rango [0,1] es interpretable como probabilidad), y nunca más para capas ocultas.

**Respuesta directa al TP:** *no todas las activaciones llegan a un resultado igualmente bueno.* La sigmoid en capas ocultas es la principal culpable de problemas: vanishing gradient, salida no centrada (dificulta la convergencia), y derivada saturable. La práctica moderna recomienda ReLU (o variantes) para ocultas, y sigmoid/softmax SOLO en la capa de salida cuando la tarea lo requiere.


### 4.5 Experimento B — Tamaño de arquitectura

Comparamos cuatro tamaños:

| Modelo | Arquitectura | Parámetros (aprox.) | Ratio param/muestras (train=120) |
|---|---|---|---|
| Pequeña | `[4]` | 35 | 0.3 |
| Mediana | `[16, 8]` | 243 | 2.0 |
| Grande | `[64, 32]` | 2.5k | 21 |
| XL | `[128, 64, 32]` | 12.5k | 104 |

**Hipótesis:** la pequeña podría sub-ajustar, la mediana es nuestro candidato, la grande y la XL deberían sobreajustar. La diferencia se va a ver en la brecha train-validation, no necesariamente en la accuracy final (porque Iris es fácil y la red termina memorizando bien).


In [ ]:
arquitecturas = {
    'pequeña [4]': (4,),
    'mediana [16,8]': (16, 8),
    'grande [64,32]': (64, 32),
    'XL [128,64,32]': (128, 64, 32),
}

historias_arq = {}
parametros_arq = {}

for nombre, arq in arquitecturas.items():
    fijar_semillas(SEED)
    modelo = construir_modelo(arquitectura=arq, activacion='relu')
    parametros_arq[nombre] = modelo.count_params()
    h = modelo.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        epochs=200,
        batch_size=16,
        callbacks=construir_callbacks(),
        verbose=0
    )
    historias_arq[nombre] = h
    eps = len(h.history['loss'])
    train_acc = h.history['accuracy'][-1]
    val_acc = h.history['val_accuracy'][-1]
    brecha = train_acc - val_acc
    print(f"{nombre:18s}  params={parametros_arq[nombre]:5d}  "
          f"epocas={eps:3d}  train_acc={train_acc:.3f}  val_acc={val_acc:.3f}  brecha={brecha:+.3f}")


In [ ]:
graficar_historia(historias_arq, 'Experimento B - Tamaño de arquitectura')


In [ ]:
# Ratio parámetros/muestras y diagnóstico visual.
muestras_train = int(0.8 * len(X_train))  # 80% del train se usa para entrenar (resto va a val_split)
df_arq = pd.DataFrame({
    'arquitectura': list(parametros_arq.keys()),
    'parametros': list(parametros_arq.values()),
})
df_arq['ratio_param/muestras'] = (df_arq['parametros'] / muestras_train).round(1)
df_arq['val_acc_final'] = [historias_arq[k].history['val_accuracy'][-1]
                           for k in df_arq['arquitectura']]
df_arq['brecha_train_val'] = [
    historias_arq[k].history['accuracy'][-1] - historias_arq[k].history['val_accuracy'][-1]
    for k in df_arq['arquitectura']
]
df_arq


**Lectura del Experimento B — el dato clave.**

El ratio parámetros/muestras es ilustrativo: la red XL tiene **~100 parámetros por cada muestra de entrenamiento**. Eso es claramente sobreparametrizado. Sin embargo, el accuracy final no necesariamente colapsa, porque:

1. Iris es un problema demasiado fácil: cualquier red con capacidad suficiente lo memoriza.
2. EarlyStopping interviene y restaura los pesos del mejor momento de validación, mitigando el daño visible en accuracy.

Lo que SÍ se ve y es interpretable es la **brecha entre train_acc y val_acc** y la **forma de las curvas**: en las redes grandes, train_loss baja muy rápido a casi 0 (memorización) mientras val_loss se estanca o repunta. Eso es la firma del overfitting, aunque la métrica final esté maquillada por EarlyStopping.

**Conclusión del experimento B:** la arquitectura `[16, 8]` es el sweet spot. Más grande no aporta accuracy y agrega varianza; más pequeña corre el riesgo de sub-ajuste. Para el resto del notebook, esta es nuestra arquitectura final.


### 4.6 Modelo final — `[16, 8]` ReLU + callbacks

Re-entrenamos el modelo final con la mejor arquitectura que encontramos, y guardamos el history para el análisis posterior. Esta vez evaluamos directamente sobre el **test set** (no sobre validation), que no se tocó en ningún momento del proceso.


In [ ]:
fijar_semillas(SEED)
modelo_final = construir_modelo(arquitectura=(16, 8), activacion='relu')

hist_final = modelo_final.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=16,
    callbacks=construir_callbacks(),
    verbose=0
)

# Predicción en test.
y_pred_proba_rna = modelo_final.predict(X_test_scaled, verbose=0)
y_pred_rna = y_pred_proba_rna.argmax(axis=1)
acc_rna = accuracy_score(y_test, y_pred_rna)
print(f"Accuracy en test: {acc_rna:.4f}  ({(y_test == y_pred_rna).sum()}/{len(y_test)} correctas)")


## 5. Evaluación de la RNA

### 5.1 Matriz de confusión y classification report

La accuracy global ya nos da una idea del desempeño. Pero como el TP pregunta por *consecuencias por clase*, miramos la matriz de confusión: nos interesa especialmente la confusión entre versicolor y virginica, que era nuestra hipótesis del EDA.


In [ ]:
# Curvas del modelo final.
graficar_historia({'modelo final [16,8] ReLU': hist_final}, 'Modelo final - curvas')

# Matriz de confusión sobre el TEST set.
cm = confusion_matrix(y_test, y_pred_rna)
fig, ax = plt.subplots(figsize=(5.5, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title(f'Matriz de confusión - RNA [16,8] - acc={acc_rna:.3f}')
plt.tight_layout()
plt.show()


In [ ]:
print(classification_report(y_test, y_pred_rna, target_names=iris.target_names, digits=3))


**Lectura.** Si nuestra hipótesis del EDA fue correcta, los errores se concentran en versicolor↔virginica. Setosa, al ser linealmente separable, debería tener precision y recall = 1.0.

### 5.2 Estabilidad entre semillas — la pregunta clave del TP

El TP pregunta literalmente: *"Si un modelo obtiene 96% y otro 100%, ¿es realmente mejor el segundo? ¿Cómo justifican esas fluctuaciones?"*

Con 30 muestras de test, la accuracy se mueve en saltos de 1/30 = 3.3%. Una diferencia 96% → 100% son apenas **una muestra distinta clasificada bien**. Antes de declarar un modelo "mejor" hay que medir cuánta varianza tiene la métrica frente a cambios de semilla.

Repetimos el entrenamiento con 5 semillas distintas y reportamos media ± desvío.


In [ ]:
semillas = [1, 7, 13, 42, 2024]
accs_por_semilla = []

for s in semillas:
    fijar_semillas(s)
    # Importante: usamos el MISMO split, solo cambia la inicialización de la red.
    # Si quisiéramos también variar el split, el rango de variación sería aún mayor.
    modelo = construir_modelo(arquitectura=(16, 8), activacion='relu')
    modelo.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        epochs=200,
        batch_size=16,
        callbacks=construir_callbacks(),
        verbose=0
    )
    pred = modelo.predict(X_test_scaled, verbose=0).argmax(axis=1)
    acc = accuracy_score(y_test, pred)
    accs_por_semilla.append(acc)
    print(f"Semilla {s:5d}  ->  test accuracy = {acc:.4f}")

import numpy as np
media_acc = np.mean(accs_por_semilla)
std_acc = np.std(accs_por_semilla)
print(f"\nMedia ± desvío: {media_acc:.4f} ± {std_acc:.4f}")
print(f"Rango: [{min(accs_por_semilla):.4f}, {max(accs_por_semilla):.4f}]")


**Interpretación.** El valor de la celda anterior es lo que hay que reportar como performance "real" de la RNA: la media, no un número puntual.

Si la media es ~0.96 y el desvío es ~0.02, una corrida que da 100% no es "mejor" que una de 96%; es la misma performance dentro del ruido del muestreo. Pretender lo contrario es **selección post-hoc** y es un error metodológico clásico.

Esto tiene una consecuencia práctica: en datasets pequeños no debería elegirse el modelo "ganador" mirando un único score; conviene comparar **distribuciones** de scores (por ejemplo, vía cross-validation o múltiples semillas).


## 6. Comparación con modelos clásicos

### 6.1 Panorámica rápida con LazyClassifier

`lazypredict` corre ~30 clasificadores con configuración por defecto y devuelve un ranking. Sirve para detectar cuáles familias de modelos vale la pena profundizar. **No reemplaza un análisis serio** (no hay tuning, no hay cross-validation), pero da un punto de partida en segundos.

Si la instalación falla en el entorno, hacemos un fallback manual con varios clasificadores de sklearn.


In [ ]:
# Intento robusto de cargar lazypredict.
lazy_disponible = False
try:
    from lazypredict.Supervised import LazyClassifier
    lazy_disponible = True
except ImportError:
    print("lazypredict no encontrado. Instalando...")
    import subprocess, sys
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                               '-q', 'lazypredict'])
        from lazypredict.Supervised import LazyClassifier
        lazy_disponible = True
    except Exception as e:
        print(f"No se pudo instalar lazypredict ({type(e).__name__}: {e}).")
        print("Vamos a hacer un panel manual equivalente.")
        lazy_disponible = False

print(f"\nLazyClassifier disponible: {lazy_disponible}")


In [ ]:
if lazy_disponible:
    # LazyClassifier maneja su propio escalado interno para varios modelos,
    # pero le pasamos los datos ya escalados para coherencia con la RNA.
    clf_lazy = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    modelos_lazy, _ = clf_lazy.fit(
        X_train_scaled, X_test_scaled, y_train, y_test
    )
    # Ordenamos por accuracy y mostramos el top-10.
    print(modelos_lazy.sort_values('Accuracy', ascending=False).head(10))
else:
    # Fallback manual: corremos varios modelos a defaults sobre el mismo split.
    candidatos_fallback = {
        'LogisticRegression': LogisticRegression(max_iter=2000, random_state=SEED),
        'LinearDiscriminantAnalysis': LinearDiscriminantAnalysis(),
        'QuadraticDiscriminantAnalysis': QuadraticDiscriminantAnalysis(),
        'KNeighborsClassifier': KNeighborsClassifier(),
        'DecisionTreeClassifier': DecisionTreeClassifier(random_state=SEED),
        'RandomForestClassifier': RandomForestClassifier(random_state=SEED),
        'GradientBoostingClassifier': GradientBoostingClassifier(random_state=SEED),
        'SVC_rbf': SVC(kernel='rbf', random_state=SEED),
        'SVC_linear': SVC(kernel='linear', random_state=SEED),
        'GaussianNB': GaussianNB(),
    }
    rows = []
    for nombre, m in candidatos_fallback.items():
        t0 = time.time()
        m.fit(X_train_scaled, y_train)
        t = time.time() - t0
        acc = accuracy_score(y_test, m.predict(X_test_scaled))
        rows.append({'Modelo': nombre, 'Accuracy_test': acc, 'Tiempo (s)': t})
    df_lazy_fb = pd.DataFrame(rows).sort_values('Accuracy_test', ascending=False)
    print(df_lazy_fb.to_string(index=False))


### 6.2 Profundización con Cross-Validation estratificada

El holdout de test (30 muestras) es ruidoso. Hacemos **5-fold CV estratificada** sobre el train set para tener una estimación más estable. Para cada modelo armamos un Pipeline que incluye el scaler, así evitamos data leakage durante el CV (el scaler se fitea solo en los folds de entrenamiento de cada iteración).

Elegimos un grupo amplio de candidatos para cubrir distintas familias:

- **Lineales**: Logistic Regression, LDA.
- **Margen**: SVM con kernel lineal y RBF.
- **Distancias**: KNN.
- **Árboles**: Decision Tree, Random Forest, Gradient Boosting.
- **Probabilísticos**: Gaussian Naive Bayes, QDA.


In [ ]:
def make_pipeline(model):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', model)
    ])

candidatos = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=SEED),
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(random_state=SEED),
    'SVM lineal': SVC(kernel='linear', random_state=SEED),
    'SVM RBF': SVC(kernel='rbf', random_state=SEED),
    'Gaussian NB': GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
filas = []

for nombre, modelo in candidatos.items():
    pipe = make_pipeline(modelo)
    t0 = time.time()
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    t_cv = time.time() - t0
    # Ajuste sobre todo el train para evaluar en test.
    pipe.fit(X_train, y_train)
    acc_test = accuracy_score(y_test, pipe.predict(X_test))
    filas.append({
        'Modelo': nombre,
        'CV mean': scores.mean(),
        'CV std': scores.std(),
        'Test acc': acc_test,
        'Tiempo CV (s)': t_cv,
    })

df_clasicos = pd.DataFrame(filas).sort_values('CV mean', ascending=False).reset_index(drop=True)
df_clasicos.style.format({
    'CV mean': '{:.4f}', 'CV std': '{:.4f}',
    'Test acc': '{:.4f}', 'Tiempo CV (s)': '{:.3f}'
}).background_gradient(cmap='Greens', subset=['CV mean', 'Test acc'])


In [ ]:
# Tabla final consolidada: clásicos + RNA.
fila_rna = pd.DataFrame([{
    'Modelo': 'RNA [16,8] ReLU',
    'CV mean': media_acc,           # media de las 5 semillas (proxy de CV)
    'CV std': std_acc,
    'Test acc': acc_rna,
    'Tiempo CV (s)': float('nan'),  # no comparable, las redes corrieron varias veces
}])

tabla_total = pd.concat([df_clasicos, fila_rna], ignore_index=True)
tabla_total = tabla_total.sort_values('CV mean', ascending=False).reset_index(drop=True)
tabla_total.style.format({
    'CV mean': '{:.4f}', 'CV std': '{:.4f}',
    'Test acc': '{:.4f}', 'Tiempo CV (s)': '{:.3f}'
}).background_gradient(cmap='Greens', subset=['CV mean'])


**Lectura.** Lo que esperamos ver:

- Los modelos top suelen estar todos dentro del rango 95-98% de CV mean. La diferencia entre el primero y el quinto es típicamente **menor que el desvío estándar** del propio CV. En otras palabras: con 150 muestras, *no se puede declarar un ganador estadístico claro*.
- La RNA, con 5 corridas, queda en el mismo rango. **No supera a los clásicos**, y entrenarla cuesta órdenes de magnitud más tiempo.
- La regresión logística y LDA, modelos extremadamente simples, suelen estar entre los top 3. Esto se entiende porque setosa es linealmente separable y la frontera versicolor/virginica es casi lineal en (petal length, petal width).

**Implicación de ingeniería:** para 150 muestras y 4 features, el modelo a poner en producción es uno clásico. Tiene mejor reproducibilidad, mejor explicabilidad, entrenamiento instantáneo, y mantiene el mismo rendimiento.


## 7. Visualización de fronteras de decisión

Para hacer tangible la diferencia entre familias de modelos, proyectamos el problema a 2D usando las dos features más informativas (petal length, petal width) y visualizamos cómo cada clasificador particiona el plano.

Esto ilumina varias cosas:

- *Forma* de la frontera (lineal, cuadrática, irregular).
- *Confianza* del modelo en cada región (qué tan abruptos son los cambios de clase).
- Por qué un modelo simple es competitivo aquí: la frontera "natural" del problema es casi lineal.


In [ ]:
def graficar_fronteras(modelos, X, y, titulo='Fronteras de decisión'):
    """Reentrena cada modelo con SOLO 2 features (petal length, petal width)
    y dibuja la frontera de decisión."""
    # Indices: petal length=2, petal width=3 (orden de iris.feature_names).
    X2 = X[:, [2, 3]]

    n = len(modelos)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4.2*rows))
    axes = np.atleast_1d(axes).flatten()

    # Mesh para la frontera.
    h = 0.02
    x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
    y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    cmap = plt.cm.RdYlBu

    last_i = -1
    for i, (nombre, mdl) in enumerate(modelos.items()):
        # Pipeline con scaler para que sea coherente con cómo entrenamos antes.
        pipe = Pipeline([('sc', StandardScaler()), ('clf', mdl)])
        pipe.fit(X2, y)
        Z = pipe.predict(grid).reshape(xx.shape)
        ax = axes[i]
        ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap)
        ax.scatter(X2[:, 0], X2[:, 1], c=y, cmap=cmap, edgecolor='k', s=35)
        ax.set_title(nombre, fontsize=11)
        ax.set_xlabel('petal length (cm)')
        ax.set_ylabel('petal width (cm)')
        last_i = i

    # Apagar ejes sobrantes si hubo más slots que modelos.
    for j in range(last_i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(titulo, fontsize=13)
    plt.tight_layout()
    plt.show()


modelos_2d = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=SEED),
    'LDA': LinearDiscriminantAnalysis(),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=SEED),
    'SVM RBF': SVC(kernel='rbf', random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=SEED),
}
graficar_fronteras(modelos_2d, iris.data, iris.target,
                   'Fronteras de decisión en (petal length, petal width)')


**Lectura.** Lo que se ve en este gráfico responde el resto del notebook:

- **Logistic Regression y LDA** trazan rectas. Y son suficientes. La frontera "real" del problema es prácticamente lineal en este plano.
- **KNN y Decision Tree** trazan fronteras irregulares, con bordes en escalera. Son modelos de alta varianza; capturan ruido del entrenamiento. Funcionan bien en Iris pero la frontera no parece "natural".
- **SVM RBF y Random Forest** suavizan, equilibrando flexibilidad y regularización. Son los modelos más prolijos visualmente.

**Para un problema cuya frontera natural es lineal, un modelo lineal es la elección óptima**: igual de preciso, mucho más interpretable, mucho más rápido. Una RNA traza fronteras parecidas al SVM RBF (no lineales suaves), pero ese poder no aporta nada cuando el problema no lo necesita.


## 8. Curva de aprendizaje del mejor clásico

Una pregunta natural al final del análisis es: **¿con más datos mejoraríamos el modelo?** O dicho de otra forma: ¿estamos en el techo del problema o estamos limitados por los 150 ejemplares disponibles?

Lo respondemos con una **curva de aprendizaje** sobre el mejor clásico (típicamente Logistic Regression o SVM): graficamos accuracy vs cantidad de muestras de entrenamiento. Si las curvas de train y validation convergen y se aplanan, ya estamos en el techo. Si aún tienen pendiente al final, más datos ayudarían.


In [ ]:
# Tomamos el modelo top de la tabla `df_clasicos` (con CV).
mejor_clasico_nombre = df_clasicos.iloc[0]['Modelo']
mejor_clasico_modelo = candidatos[mejor_clasico_nombre]
print(f"Mejor clásico segun CV: {mejor_clasico_nombre}")

pipe_mejor = make_pipeline(mejor_clasico_modelo)
sizes, train_scores, val_scores = learning_curve(
    pipe_mejor, iris.data, iris.target,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=cv,
    scoring='accuracy',
    n_jobs=1,
    shuffle=True,
    random_state=SEED,
)

train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
val_mean, val_std = val_scores.mean(axis=1), val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, train_mean, 'o-', color='teal', label='Train')
ax.fill_between(sizes, train_mean-train_std, train_mean+train_std, alpha=0.2, color='teal')
ax.plot(sizes, val_mean, 'o-', color='orange', label='Validation (CV)')
ax.fill_between(sizes, val_mean-val_std, val_mean+val_std, alpha=0.2, color='orange')
ax.set_xlabel('Cantidad de muestras de entrenamiento')
ax.set_ylabel('Accuracy')
ax.set_title(f'Curva de aprendizaje - {mejor_clasico_nombre}')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Lectura.** Si las curvas convergen y la curva de validación se aplana cerca de su máximo, recolectar más muestras *no produciría una mejora significativa*. Si las dos curvas siguen separadas y val tiene pendiente positiva al final, más datos ayudarían.

En Iris, lo esperable es ver convergencia clara: el problema es simple y 150 muestras alcanzan. La conclusión clínica/operativa sería "no gastar recursos recolectando más ejemplares; la limitación no está en los datos".


## 9. Conclusiones y respuestas a las consignas del TP

A continuación respondemos cada premisa que el TP plantea sobre el Ejercicio 2, con base en lo observado en este notebook.

### ¿Todas las funciones de activación permiten llegar a un resultado aceptable? ¿Cuáles presentan problemas y por qué?

**No todas.** ReLU, ELU, Leaky ReLU y Tanh llegan a un buen resultado en este problema (porque la red es chica y el problema simple). **Sigmoid** es la que peor se comporta como activación de capa oculta: su derivada satura cerca de 0 fuera del intervalo (-2, 2), produciendo *vanishing gradient*. Su salida no centrada en cero también penaliza la convergencia. Por eso la práctica moderna reserva sigmoid exclusivamente para la capa de salida en clasificación binaria, y nunca para capas ocultas. Para multiclase, la salida se hace con softmax.

### ¿Con cuántas capas ocultas y qué dimensiones se logra entrenar la red?

`[16, 8]` con activación ReLU es un sweet spot. Una sola capa de 4 neuronas también funciona (Iris es muy simple) pero arriesga sub-ajuste si la inicialización es desafortunada. Arquitecturas grandes (`[64,32]` o `[128,64,32]`) sobreajustan: sus curvas muestran train_loss colapsando a ~0 mientras val_loss se estanca o repunta, y tienen ~20 a ~100 parámetros por cada muestra de entrenamiento. EarlyStopping mitiga el daño visible pero no es una excusa para usar redes innecesariamente grandes.

### ¿Cómo se codifican las clases en la capa de salida? ¿Existe más de una forma válida?

Hay dos formas válidas y matemáticamente equivalentes:

| Codificación | Forma de y | Loss compatible |
|---|---|---|
| Label encoding | `(N,)` con enteros 0/1/2 | `sparse_categorical_crossentropy` |
| One-Hot | `(N, 3)` con vectores unitarios | `categorical_crossentropy` |

La capa de salida es **siempre** `Dense(3, activation='softmax')`. Las 3 neuronas representan las 3 clases; softmax normaliza las salidas a una distribución de probabilidad.

**El resultado final no cambia.** Es una conveniencia de implementación: si el target ya está como entero, `sparse_categorical_crossentropy` ahorra una transformación. Si se necesita trabajar con probabilidades por clase (por ejemplo para *label smoothing*), conviene one-hot.

### ¿Cómo se justifica que la RNA es robusta dado que el dataset tiene 150 muestras?

**Estrictamente: no es robusta**, en el sentido de que su métrica fluctúa con la semilla. Pero podemos hacerla razonablemente confiable así:

- Reportar **media ± desvío sobre múltiples semillas**, no un score puntual.
- Usar arquitectura pequeña (parámetros del orden de las muestras de entrenamiento).
- EarlyStopping con `restore_best_weights` para no entregar nunca una red sobreajustada.
- Verificar que la accuracy final cae dentro del rango de los clásicos. Si fuera mucho más baja, hay un problema de implementación; si fuera mucho más alta, probablemente haya leakage.

### ¿Qué tan estables son las métricas entre semillas? Si un modelo da 96% y otro 100%, ¿es realmente mejor el segundo?

**No, no es realmente mejor.** Con 30 muestras de test, la accuracy se mueve en saltos discretos de 1/30 ≈ 3.3%. Ir de 96.6% a 100% son *exactamente* 1 muestra. Esa diferencia está dentro del ruido de muestreo. La forma correcta de comparar dos modelos es por sus distribuciones de score (cross-validation o múltiples semillas), no por un único número puntual.

### Comparación con clásicos: diferencias en métricas, tiempo y facilidad de configuración

- **Métrica:** prácticamente empatada. La diferencia entre el mejor clásico y la RNA cae dentro del desvío estándar de la cross-validation.
- **Tiempo de entrenamiento:** los clásicos tardan milisegundos. La RNA tarda decenas de segundos por corrida, y conviene hacer múltiples corridas para promediar.
- **Facilidad de configuración:** los clásicos tienen 1-3 hiperparámetros relevantes con buenos defaults. Una RNA tiene como mínimo: arquitectura (capas, neuronas), activación, optimizador, learning rate, batch size, epochs, callbacks, regularización. Cada uno puede arruinar el entrenamiento.
- **Reproducibilidad:** un clásico con `random_state` fijo da exactamente el mismo modelo. Una RNA, incluso con todas las semillas fijas, puede tener variabilidad por operaciones no deterministas en GPU.
- **Explicabilidad:** Logistic Regression entrega coeficientes interpretables. Decision Tree entrega reglas legibles. La RNA es una caja negra que requiere herramientas adicionales (SHAP/LIME) para explicar predicciones individuales.

### ¿Qué camino conviene cuando hay pocas observaciones?

**Machine Learning clásico, sin dudas.** Con menos de algunos miles de muestras, las redes neuronales:

- No tienen suficientes datos para sobreparametrizar a su favor.
- Pierden por configuración costosa.
- Igualan o quedan por debajo de modelos clásicos bien tuneados.

**Intuición a llevarse:** el deep learning brilla cuando la complejidad del problema (no del dataset) supera la capacidad de los modelos clásicos. Esto suele ocurrir cuando los datos *no son tabulares* (imágenes, audio, texto, secuencias largas) y/o cuando hay *miles a millones* de ejemplos. Para datos tabulares de pocas dimensiones y pocas muestras, los modelos clásicos siguen siendo el estándar de la industria. Recientes benchmarks (Grinsztajn et al., 2022) confirman que para datos tabulares, los modelos basados en árboles (XGBoost, LightGBM, Random Forest) siguen dominando incluso frente a transformers tabulares.

---

### Recomendación final para el jardín botánico

Para clasificar especies de Iris a partir de las cuatro mediciones morfológicas, la solución recomendada es un modelo lineal con scaler (Logistic Regression o LDA) dentro de un Pipeline de sklearn. Se entrena en milisegundos, alcanza accuracy 95-98% en cross-validation (idéntica a la RNA), entrega coeficientes interpretables que el equipo de investigación puede examinar, y se despliega como un objeto `joblib` de pocos KB. La RNA, en este contexto, sería una sobre-ingeniería injustificada: más complejidad, más mantenimiento, más infraestructura, sin ganancia de desempeño.
